## Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [2]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 52.67 GB
MemAvailable: 851.61 GB
Free GPU Memory (GB): 39.3936

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successfully authenticated with the Hugging Face 

## 2. Generating the cmw.txt file

In [ ]:
exp_id = "09-03-1"

In [ ]:
def process_file(file_path):
    # Read the file and count the number of rows
    with open(file_path, 'r') as file:
        lines = file.readlines()
    
    print(f"Number of rows: {len(lines)}")
    
    processed_lines = []

    # Process each line
    for line_idx, line in enumerate(lines):
        if line_idx > 10:
            break
        # Remove the trailing backslash and newline characters
        line = line.strip().rstrip('\\')
        
        # Split the line into the incorrect word and the correct words
        if '->' in line:
            incorrect_word, correct_words = line.split('->')
            
            # Split the correct words by comma and strip whitespace
            correct_words_list = [word.strip() for word in correct_words.split(',')]
            
            # Generate the output lines
            for correct_word in correct_words_list:
                processed_lines.append(f"{correct_word} {incorrect_word}")
        
    
    # Print each processed line
    for processed_line in processed_lines:
        print(processed_line)

# Define the path to your text file
file_path = "/nfs/homedirs/daro/git/quantization-reliability/data/cmw.txt"

# Run the function
process_file(file_path)

Number of rows: 4312
abandoned abandonned
aberration aberation
abilities abilityes
abilities abilties
ability abilty
abandon abondon
about abbout
about abotu
about a abouta
about it aboutit
about the aboutthe


In [ ]:
def process_and_save_file(input_file_path, output_file_path):
    # Read the input file and count the number of rows
    with open(input_file_path, 'r') as file:
        lines = file.readlines()
    
    print(f"Number of rows: {len(lines)}")
    
    processed_lines = []

    # Process each line
    for line in lines:
        # Remove the trailing backslash and newline characters
        line = line.strip().rstrip('\\')
        
        # Split the line into the incorrect word and the correct words
        if '->' in line:
            incorrect_word, correct_words = line.split('->')
            
            # Split the correct words by comma and strip whitespace
            correct_words_list = [word.strip() for word in correct_words.split(',')]
            
            # Generate the output lines
            for correct_word in correct_words_list:
                processed_lines.append(f"{correct_word}: {incorrect_word}")
    
    # Write the processed lines to the output file
    with open(output_file_path, 'w') as output_file:
        for processed_line in processed_lines:
            output_file.write(processed_line + '\n')
    
    print(f"Processed lines have been saved to {output_file_path}")

# Define the paths to your text files
input_file_path = 'data/cmw.txt'
output_file_path = 'data/cmw_v2.txt'

# Run the function
process_and_save_file(input_file_path, output_file_path)

Number of rows: 4312
Processed lines have been saved to data/cmw_v2.txt


In [ ]:
def load_file_to_dict(file_path):
    cmw_dict = {}

    # Read the file and populate the dictionary
    with open(file_path, 'r') as file:
        lines = file.readlines()
        
        for line in lines:
            # Split each line by space to get the correct and incorrect words
            correct_word, incorrect_word = line.strip().split(':')
            cmw_dict[correct_word.strip()] = incorrect_word.strip()

    return cmw_dict

# Define the path to your processed text file
processed_file_path = 'data/cmw_v2.txt'

# Load the file into a dictionary
cmw_dict = load_file_to_dict(processed_file_path)

# Print the dictionary to verify
print(cmw_dict)

{'abandoned': 'abondoned', 'aberration': 'aberation', 'abilities': 'abilties', 'ability': 'abilty', 'abandon': 'adbandon', 'about': 'boaut', 'about a': 'abouta', 'about it': 'aboutit', 'about the': 'aboutthe', 'absence': 'absense', 'abandoning': 'abondoning', 'abandons': 'abondons', 'aborigine': 'aborigene', 'accessories': 'accesories', 'accident': 'acident', 'abortifacient': 'abortificant', 'abbreviate': 'abreviate', 'abbreviated': 'abreviated', 'abbreviation': 'abreviation', 'arbitrary': 'arbitary', 'abseil': 'absail', 'abseiling': 'absailing', 'absolutely': 'absolutly', 'absorption': 'absorbtion', 'abundance': 'abudance', 'abundances': 'abundancies', 'abundant': 'abundunt', 'abuts': 'abutts', 'academy': 'accademy', 'academic': 'acedemic', 'accused': 'acused', 'acceleration': 'accelleration', 'accession': 'accension', 'ascension': 'accension', 'acceptance': 'acceptence', 'acceptable': 'acceptible', 'accessible': 'accessable', 'accidentally': 'accidently', 'acclimatization': 'acclimit

In [12]:
from nltk.corpus import wordnet

synonyms = wordnet.synsets("square")

In [13]:
synonyms = []
for syn in wordnet.synsets("square"):
     for l in syn.lemmas():
          synonyms.append(l.name())
               
print(set(synonyms))

{'hearty', 'straight', 'straightforward', 'foursquare', 'lame', 'square_up', 'feather', 'substantial', 'squarely', 'public_square', 'second_power', 'square', 'satisfying', 'solid', 'square_toes'}


## 3. Generating Typos

In [3]:
emoji_dict = {
    "smile": "😊",
    "smiles": "😊",
    "smiling": "😊",
    "laugh": "😂",
    "laughs": "😂",
    "laughing": "😂",
    "sad": "😢",
    "sadness": "😢",
    "crying": "😢",
    "angry": "😠",
    "angered": "😠",
    "anger": "😠",
    "love": "❤️",
    "loves": "❤️",
    "loving": "❤️",
    "heart": "❤️",
    "hearts": "❤️",
    "sun": "☀️",
    "sunny": "☀️",
    "sunshine": "☀️",
    "moon": "🌙",
    "moonlight": "🌙",
    "star": "⭐",
    "stars": "⭐",
    "starry": "⭐",
    "rain": "🌧️",
    "rains": "🌧️",
    "rainy": "🌧️",
    "cloud": "☁️",
    "clouds": "☁️",
    "cloudy": "☁️",
    "snow": "❄️",
    "snowy": "❄️",
    "snowing": "❄️",
    "fire": "🔥",
    "fiery": "🔥",
    "hot": "🔥",
    "cold": "🥶",
    "freezing": "🥶",
    "ice": "🧊",
    "icy": "🧊",
    "tree": "🌳",
    "trees": "🌳",
    "forest": "🌳",
    "flower": "🌸",
    "flowers": "🌸",
    "flowering": "🌸",
    "dog": "🐶",
    "dogs": "🐶",
    "cat": "🐱",
    "cats": "🐱",
    "bird": "🐦",
    "birds": "🐦",
    "fish": "🐠",
    "fishes": "🐠",
    "car": "🚗",
    "cars": "🚗",
    "bus": "🚌",
    "buses": "🚌",
    "train": "🚂",
    "trains": "🚂",
    "airplane": "✈️",
    "airplanes": "✈️",
    "flying": "✈️",
    "boat": "🚢",
    "boats": "🚢",
    "sailing": "🚢",
    "house": "🏠",
    "houses": "🏠",
    "building": "🏢",
    "buildings": "🏢",
    "food": "🍔",
    "eat": "🍔",
    "eating": "🍔",
    "drink": "🥤",
    "drinks": "🥤",
    "drinking": "🥤",
    "music": "🎵",
    "musical": "🎵",
    "song": "🎵",
    "book": "📚",
    "books": "📚",
    "reading": "📚",
    "movie": "🎬",
    "movies": "🎬",
    "film": "🎬",
    "phone": "📱",
    "phones": "📱",
    "calling": "📱",
    "computer": "💻",
    "computers": "💻",
    "computing": "💻",
    "money": "💰",
    "rich": "💰",
    "wealth": "💰",
    "clock": "🕰️",
    "clocks": "🕰️",
    "time": "🕰️",
    "gift": "🎁",
    "gifts": "🎁",
    "present": "🎁",
    "presents": "🎁",
    "birthday": "🎂",
    "birthdays": "🎂",
    "celebrate": "🎉",
    "celebrating": "🎉",
    "celebration": "🎉",
    "win": "🏆",
    "winning": "🏆",
    "winner": "🏆",
    "lose": "😞",
    "losing": "😞",
    "loser": "😞",
    "sport": "⚽",
    "sports": "⚽",
    "athletic": "⚽",
    "sleep": "😴",
    "sleeping": "😴",
    "sleepy": "😴",
    "work": "💼",
    "working": "💼",
    "job": "💼",
    "school": "🏫",
    "studying": "🏫",
    "learn": "🏫",
    "vacation": "🏖️",
    "vacationing": "🏖️",
    "holiday": "🏖️",
    "travel": "✈️",
    "traveling": "✈️",
    "journey": "✈️",
    "idea": "💡",
    "ideas": "💡",
    "thinking": "💡",
    "question": "❓",
    "questions": "❓",
    "questioning": "❓",
    "answer": "✅",
    "answers": "✅",
    "answering": "✅",
    "warning": "⚠️",
    "warnings": "⚠️",
    "caution": "⚠️",
    "stop": "🛑",
    "stopping": "🛑",
    "halt": "🛑",
    "go": "✅",
    "going": "✅",
    "start": "✅",
    "hello": "👋",
    "hi": "👋",
    "greeting": "👋",
    "goodbye": "👋",
    "bye": "👋",
    "farewell": "👋",
    "yes": "👍",
    "agree": "👍",
    "agreeing": "👍",
    "no": "👎",
    "disagree": "👎",
    "disagreeing": "👎",
    "ok": "👌",
    "okay": "👌",
    "fine": "👌",
    "good": "😊",
    "great": "😊",
    "excellent": "😊",
    "bad": "😞",
    "terrible": "😞",
    "awful": "😞",
    "pizza": "🍕",
    "pizzas": "🍕",
    "hamburger": "🍔",
    "hamburgers": "🍔",
    "burger": "🍔",
    "burgers": "🍔",
    "fries": "🍟",
    "french fries": "🍟",
    "sushi": "🍣",
    "rice": "🍚",
    "noodles": "🍜",
    "ramen": "🍜",
    "taco": "🌮",
    "tacos": "🌮",
    "burrito": "🌯",
    "burritos": "🌯",
    "egg": "🥚",
    "eggs": "🥚",
    "bread": "🍞",
    "sandwich": "🥪",
    "sandwiches": "🥪",
    "cake": "🎂",
    "cakes": "🎂",
    "cookie": "🍪",
    "cookies": "🍪",
    "candy": "🍬",
    "candies": "🍬",
    "lollipop": "🍭",
    "lollipops": "🍭",
    "ice cream": "🍦",
    "coffee": "☕",
    "tea": "🍵",
    "milk": "🥛",
    "beer": "🍺",
    "beers": "🍺",
    "wine": "🍷",
    "cocktail": "🍸",
    "cocktails": "🍸",
    "fruit": "🍎",
    "fruits": "🍎",
    "apple": "🍎",
    "apples": "🍎",
    "banana": "🍌",
    "bananas": "🍌",
    "orange": "🍊",
    "oranges": "🍊",
    "lemon": "🍋",
    "lemons": "🍋",
    "strawberry": "🍓",
    "strawberries": "🍓",
    "watermelon": "🍉",
    "watermelons": "🍉",
    "grapes": "🍇",
    "pineapple": "🍍",
    "pineapples": "🍍",
    "peach": "🍑",
    "peaches": "🍑",
    "cherry": "🍒",
    "cherries": "🍒",
    "vegetable": "🥕",
    "vegetables": "🥕",
    "carrot": "🥕",
    "carrots": "🥕",
    "broccoli": "🥦",
    "tomato": "🍅",
    "tomatoes": "🍅",
    "potato": "🥔",
    "potatoes": "🥔",
    "corn": "🌽",
    "mushroom": "🍄",
    "mushrooms": "🍄",
    "avocado": "🥑",
    "avocados": "🥑",
    "eggplant": "🍆",
    "eggplants": "🍆",
    "cucumber": "🥒",
    "cucumbers": "🥒",
    "pepper": "🌶️",
    "peppers": "🌶️",
    "garlic": "🧄",
    "onion": "🧅",
    "onions": "🧅",
    "peanut": "🥜",
    "peanuts": "🥜",
    "chestnut": "🌰",
    "chestnuts": "🌰",
    "bacon": "🥓",
    "salad": "🥗",
    "salads": "🥗",
    "popcorn": "🍿",
    "butter": "🧈",
    "salt": "🧂",
    "salty": "🧂",
    "brain": "🧠",
    "brains": "🧠",
    "bone": "🦴",
    "bones": "🦴",
    "eyes": "👀",
    "eye": "👁️",
    "ear": "👂",
    "ears": "👂",
    "nose": "👃",
    "noses": "👃",
    "mouth": "👄",
    "mouths": "👄",
    "tongue": "👅",
    "tongues": "👅",
    "baby": "👶",
    "babies": "👶",
    "child": "🧒",
    "children": "🧒",
    "boy": "👦",
    "boys": "👦",
    "girl": "👧",
    "girls": "👧",
    "man": "👨",
    "men": "👨",
    "woman": "👩",
    "women": "👩",
    "person": "🧑",
    "people": "🧑",
    "family": "👪",
    "families": "👪",
    "couple": "👫",
    "couples": "👫",
    "bride": "👰",
    "groom": "🤵",
    "grandma": "👵",
    "grandmother": "👵",
    "grandpa": "👴",
    "grandfather": "👴",
    "princess": "👸",
    "prince": "🤴",
    "santa": "🎅",
    "ghost": "👻",
    "ghosts": "👻",
    "alien": "👽",
    "aliens": "👽",
    "robot": "🤖",
    "robots": "🤖",
    "zombie": "🧟",
    "zombies": "🧟",
    "footprint": "👣",
    "footprints": "👣",
    "monkey": "🐵",
    "monkeys": "🐵",
    "gorilla": "🦍",
    "gorillas": "🦍",
    "orangutan": "🦧",
    "orangutans": "🦧",
    "panda": "🐼",
    "pandas": "🐼",
    "sloth": "🦥",
    "sloths": "🦥",
    "otter": "🦦",
    "otters": "🦦",
    "skunk": "🦨",
    "skunks": "🦨",
    "kangaroo": "🦘",
    "kangaroos": "🦘",
    "badger": "🦡",
    "badgers": "🦡",
    "paw": "🐾",
    "paws": "🐾",
    "turkey": "🦃",
    "turkeys": "🦃",
    "chicken": "🐔",
    "chickens": "🐔",
    "rooster": "🐓",
    "roosters": "🐓",
    "penguin": "🐧",
    "penguins": "🐧",
    "dove": "🕊️",
    "doves": "🕊️",
    "eagle": "🦅",
    "eagles": "🦅",
    "duck": "🦆",
    "ducks": "🦆",
    "swan": "🦢",
    "swans": "🦢",
    "owl": "🦉",
    "owls": "🦉",
    "flamingo": "🦩",
    "flamingos": "🦩",
    "peacock": "🦚",
    "peacocks": "🦚",
    "parrot": "🦜",
    "parrots": "🦜",
    "frog": "🐸",
    "frogs": "🐸",
    "crocodile": "🐊",
    "crocodiles": "🐊",
    "turtle": "🐢",
    "turtles": "🐢",
    "lizard": "🦎",
    "lizards": "🦎",
    "snake": "🐍",
    "snakes": "🐍",
    "dragon": "🐉",
    "dragons": "🐉",
    "dinosaur": "🦕",
    "dinosaurs": "🦕",
    "whale": "🐳",
    "whales": "🐳",
    "dolphin": "🐬",
    "dolphins": "🐬",
    "seal": "🦭",
    "seals": "🦭",
    "shark": "🦈",
    "sharks": "🦈",
    "octopus": "🐙",
    "octopuses": "🐙",
    "shell": "🐚",
    "shells": "🐚",
    "coral": "🪸",
    "corals": "🪸",
    "butterfly": "🦋",
    "butterflies": "🦋",
    "bug": "🐛",
    "bugs": "🐛",
    "ant": "🐜",
    "ants": "🐜",
    "honeybee": "🐝",
    "honeybees": "🐝",
    "ladybug": "🐞",
    "ladybugs": "🐞",
    "cricket": "🦗",
    "crickets": "🦗",
    "spider": "🕷️",
    "spiders": "🕷️",
    "scorpion": "🦂",
    "scorpions": "🦂",
    "microbe": "🦠",
    "microbes": "🦠",
    "bouquet": "💐",
    "bouquets": "💐",
    "tulip": "🌷",
    "tulips": "🌷",
    "rose": "🌹",
    "roses": "🌹",
    "wilted flower": "🥀",
    "sunflower": "🌻",
    "sunflowers": "🌻",
    "blossom": "🌼",
    "blossoms": "🌼",
    "herb": "🌿",
    "herbs": "🌿",
    "shamrock": "☘️",
    "shamrocks": "☘️",
    "maple leaf": "🍁",
    "maple leaves": "🍁",
    "fallen leaf": "🍂",
    "fallen leaves": "🍂",
    "leaf fluttering": "🍃",
    "leaves fluttering": "🍃",
    "mushroom": "🍄",
    "mushrooms": "🍄",
    "cactus": "🌵",
    "cacti": "🌵",
    "palm tree": "🌴",
    "palm trees": "🌴",
    "evergreen tree": "🌲",
    "evergreen trees": "🌲",
    "deciduous tree": "🌳",
    "deciduous trees": "🌳",
    "christmas": "🎄",
    "mountain": "⛰️",
    "mountains": "⛰️",
    "volcano": "🌋",
    "volcanoes": "🌋",
    "desert": "🏜️",
    "deserts": "🏜️",
    "island": "🏝️",
    "islands": "🏝️",
    "national park": "🏞️",
    "national parks": "🏞️",
    "globe": "🌍",
    "earth": "🌎",
    "world": "🌏",
    "map": "🗺️",
    "maps": "🗺️",
    "japan": "🗾",
    "compass": "🧭",
    "compasses": "🧭",
    "snowcapped mountain": "🏔️",
    "snowcapped mountains": "🏔️",
    "camping": "🏕️",
    "beach": "🏖️",
    "beaches": "🏖️",
    "building construction": "🏗️",
    "houses": "🏘️",
    "city": "🏙️",
    "cities": "🏙️",
    "cityscapes": "🏙️",
    "derelict house": "🏚️",
    "derelict houses": "🏚️",
    "classical building": "🏛️",
    "classical buildings": "🏛️",
    "factory": "🏭",
    "factories": "🏭",
    "brick": "🧱",
    "bricks": "🧱",
    "rock": "🪨",
    "rocks": "🪨",
    "wood": "🪵",
    "woods": "🪵",
    "hut": "🛖",
    "huts": "🛖",
    "stadium": "🏟️",
    "stadiums": "🏟️",
    "ferris wheel": "🎡",
    "ferris wheels": "🎡",
    "roller coaster": "🎢",
    "roller coasters": "🎢",
    "carousel horse": "🎠",
    "carousel horses": "🎠",
    "fountain": "⛲",
    "fountains": "⛲",
    "tent": "⛺",
    "tents": "⛺",
    "foggy": "🌁",
    "night": "🌃",
    "sunrise": "🌅",
    "sunrises": "🌅",
    "sunset": "🌇",
    "sunsets": "🌇",
    "rainbow": "🌈",
    "rainbows": "🌈",
    "wave": "🌊",
    "waves": "🌊",
    "tornado": "🌪️",
    "tornados": "🌪️",
    "typhoon": "🌀",
    "typhoons": "🌀",
    "hurricane": "🌀",
    "hurricanes": "🌀",
    "fog": "🌫️",
    "wind face": "🌬️",
    "wind faces": "🌬️",
    "blowing wind": "🌬️",
    "hot spring": "♨️",
    "hot springs": "♨️",
    "thermometer": "🌡️",
    "thermometers": "🌡️",
    "drop": "💧",
    "drops": "💧",
    "sweat droplets": "💦",
    "ice": "🧊",
    "snowflake": "❄️",
    "snowflakes": "❄️",
    "snowman": "☃️",
    "snowmen": "☃️",
    "snowman without snow": "⛄",
    "comet": "☄️",
    "comets": "☄️",
    "fire": "🔥",
    "flames": "🔥",
    "jack-o-lantern": "🎃",
    "jack-o-lanterns": "🎃",
    "fireworks": "🎆",
    "sparkler": "🎇",
    "sparklers": "🎇",
    "firecracker": "🧨",
    "firecrackers": "🧨",
    "sparkles": "✨",
    "balloon": "🎈",
    "balloons": "🎈",
    "party": "🎉",
    "parties": "🎉",
    "confetti ball": "🎊",
    "confetti balls": "🎊",
    "tanabata tree": "🎋",
    "tanabata trees": "🎋",
    "pine": "🎍",
    "pine": "🎍",
    "japanese dolls": "🎎",
    "carp streamer": "🎏",
    "carp streamers": "🎏",
    "wind chime": "🎐",
    "wind chimes": "🎐",
    "moon viewing ceremony": "🎑",
    "ribbon": "🎀",
    "ribbons": "🎀",
    "gift": "🎁",
    "gifts": "🎁",
    "present": "🎁",
    "presents": "🎁",
    "ticket": "🎫",
    "tickets": "🎫",
    "medal": "🎖️",
    "medals": "🎖️",
    "trophy": "🏆",
    "trophies": "🏆",
    "1st": "🥇",
    "2nd": "🥈",
    "3rd": "🥉",
    "first": "🥇",
    "second": "🥈",
    "third": "🥉",
    "ball": "⚽",
    "balls": "⚽",
    "baseball": "⚾",
    "baseballs": "⚾",
    "softball": "🥎",
    "softballs": "🥎",
    "basketball": "🏀",
    "basketballs": "🏀",
    "volleyball": "🏐",
    "volleyballs": "🏐",
    "american": "🏈",
    "american": "🏈",
    "rugby": "🏉",
    "rugby": "🏉",
    "tennis": "🎾",
    "tennis balls": "🎾",
    "flying disc": "🥏",
    "flying discs": "🥏",
    "bowling": "🎳",
    "cricket": "🏏",
    "cricket": "🏏",
    "hockey": "🏑",
    "ice hockey": "🏒",
    "lacrosse": "🥍",
    "ping pong": "🏓",
    "badminton": "🏸",
    "boxing": "🥊",
    "boxing": "🥊",
    "goal net": "🥅",
    "goal nets": "🥅",
    "flag in hole": "⛳",
    "flags in hole": "⛳",
    "skate": "⛸️",
    "skates": "⛸️",
    "fishing": "🎣",
    "diving": "🤿",
    "running": "🎽",
    "skis": "🎿",
    "sled": "🛷",
    "sleds": "🛷",
    "curling stone": "🥌",
    "curling stones": "🥌",
    "bullseye": "🎯",
    "bullseyes": "🎯",
    "yo-yo": "🪀",
    "yo-yos": "🪀",
    "kite": "🪁",
    "kites": "🪁",
    "water pistol": "🔫",
    "water pistols": "🔫",
    "pool 8 ball": "🎱",
    "pool 8 balls": "🎱",
    "crystal ball": "🔮",
    "crystal balls": "🔮",
    "wand": "🪄",
    "wands": "🪄",
    "game": "🎮",
    "games": "🎮",
    "joystick": "🕹️",
    "joysticks": "🕹️",
    "slot machine": "🎰",
    "slot machines": "🎰",
    "die": "🎲",
    "dice": "🎲",
    "puzzle": "🧩",
    "puzzle": "🧩",
    "teddy": "🧸",
    "teddy": "🧸",
    "spade suit": "♠️",
    "heart suit": "♥️",
    "diamond suit": "♦️",
    "club suit": "♣️",
    "pawn": "♟️",
    "pawns": "♟️",
    "joker": "🃏",
    "jokers": "🃏",
    "mahjong": "🀄",
    "mahjong": "🀄",
    "flower playing cards": "🎴",
    "arts": "🎭",
    "art": "🎭",
    "picture": "🖼️",
    "pictures": "🖼️",
    "paint": "🎨",
    "paints": "🎨",
    "painter": "🎨",
    "thread": "🧵",
    "threads": "🧵",
    "needle": "🪡",
    "needles": "🪡",
    "yarn": "🧶",
    "yarns": "🧶",
    "knot": "🪢",
    "knots": "🪢",
    "mending heart": "💖",
    "mending hearts": "💖",
    "heart on fire": "❤️",
    "hearts on fire": "❤️",
    "face with spiral eyes": "😵",
    "faces with spiral eyes": "😵",
    "face in clouds": "😶",
    "faces in clouds": "😶",
    "face exhaling": "😮",
    "faces exhaling": "😮",
    "face with peeking eye": "👀",
    "faces with peeking eye": "👀",
    "saluting face": "🫡",
    "saluting faces": "🫡",
    "dotted line face": "👤",
    "dotted line faces": "👤",
    "face holding back tears": "😢",
    "faces holding back tears": "😢",
    "right": "👉",
    "left": "👈",
    "palm down hand": "🖐️",
    "palm down hands": "🖐️",
    "palm up hand": "🤲",
    "palm up hands": "🤲",
    "handshake": "🤝",
    "handshakes": "🤝",
    "heart hands": "🤗",
    "biting": "😬",
    "bite": "😬",
    "crown": "👑",
    "crowns": "👑",
    "pregnant": "🤰",
    "pregnant": "🤰",
    "troll": "👹",
    "trolls": "👹",
    "coral": "🐚",
    "corals": "🐚",
    "lotus": "🌸",
    "lotuses": "🌸",
    "nest": "🪹",
    "nests": "🪹",
    "egg": "🐣",
    "eggs": "🐣",
    "beans": "🫘",
    "liquid": "🍶",
    "jar": "🏺",
    "jars": "🏺",
    "slide": "🛝",
    "slides": "🛝",
    "wheel": "🛞",
    "wheels": "🛞",
    "buoy": "🛟",
    "buoys": "🛟",
    "hamsa": "🧿",
    "hamsas": "🧿",
    "mirror ball": "🕺",
    "mirror balls": "🕺",
    "battery": "🔋",
    "batteries": "🔋",
    "crutch": "🦯",
    "crutches": "🦯",
    "x-ray": "🩻",
    "x-rays": "🩻",
    "bubbles": "🫧",
    "card": "💳",
    "cards": "💳",
    "equals": "=",
    "equal": "=",
    "wireless": "📶",
    "khanda": "🕉️",
    "maracas": "🎵",
    "flute": "🎼",
    "flutes": "🎼",
    "hyacinth": "🌺",
    "hyacinths": "🌺",
    "jellyfish": "🐙",
    "jellyfishes": "🐙",
    "wing": "🦅",
    "wings": "🦅",
    "goose": "🦢",
    "geese": "🦢",
    "moose": "🦌",
    "donkey": "🐴",
    "donkeys": "🐴",
    "bird": "🐦",
    "birds": "🐦",
    "phoenix": "🦅",
    "phoenixes": "🦅",
    "ginger": "🥔",
    "pea pod": "🫛",
    "pea pods": "🫛",
    "folding hand fan": "🌬️",
    "folding hand fans": "🌬️",
    "hair pick": "🧼",
    "hair picks": "🧼",
}

In [4]:
char_map = {
    'A': '@',    'a': '@',
    'B': '8',    'b': '6',
    'C': '(',    'c': '©',
    'D': ')',    'd': 'ð',
    'E': '3',    'e': '€',
    'F': 'ƒ',    'f': 'ƒ',
    'G': '6',    'g': '9',
    'H': '#',    'h': '♓',
    'I': '1',    'i': '!',
    'J': '7',    'j': 'ʝ',
    'K': '|<',   'k': '|¢',
    'L': '£',    'l': '|',
    'M': 'M',    'm': 'ᴍ',
    'N': 'И',    'n': 'ñ',
    'O': '0',    'o': '°',
    'P': '?',    'p': 'ρ',
    'Q': '2',    'q': 'φ',
    'R': '®',    'r': 'Я',
    'S': '5',    's': '§',
    'T': '7',    't': '†',
    'U': 'µ',    'u': 'υ',
    'V': '√',    'v': 'ν',
    'W': 'Ш',    'w': 'ω',
    'X': '×',    'x': '⨯',
    'Y': '¥',    'y': 'ч',
    'Z': '2',    'z': 'ℤ',
    '0': 'O',
    '1': 'l',
    '2': 'Z',
    '3': 'E',
    '4': 'A',
    '5': 'S',
    '6': 'G',
    '7': 'T',
    '8': 'B',
    '9': 'g',
    '.': '·',
    ',': '`',
    ':': ';',
    ';': ':',
    '!': '¡',
    '?': '¿',
    '"': "''",
    "'": '`',
    '(': '{',
    ')': '}',
    '[': '(',
    ']': ')',
    '{': '[',
    '}': ']',
    '<': '‹',
    '>': '›',
    '/': '\\',
    '\\': '/',
    '|': 'ǀ',
    '+': '†',
    '-': '−',
    '*': '×',
    '=': '≡',
    '%': '‰',
    '$': '€',
    '&': '＆',
    '@': 'α',
    '#': '♯',
    '^': 'ˆ',
    '~': '∼',
    '`': '´'
}

In [5]:
keyboard_adjacency = {
    'a': 'qwsxzy', 'b': 'vghn', 'c': 'xdfv', 'd': 'ersfxcw', 'e': 'wrsdf34',
    'f': 'rtgvcd', 'g': 'tyuhbvf', 'h': 'yuijknbg', 'i': 'uojkl89', 'j': 'uikmnh',
    'k': 'iolmj', 'l': 'opk;', 'm': 'njk,', 'n': 'bhjm', 'o': 'iklp90',
    'p': 'ol;[0-', 'q': 'wa12', 'r': 'etdf45', 's': 'wedxzay', 't': 'ryfg56',
    'u': 'yhji78', 'v': 'cfgb', 'w': 'qase23', 'x': 'yzsdc', 'y': 'tghuj567xsaz',
    'z': 'asxytghuj765',
    '1': '2q`', '2': '13qw', '3': '24we', '4': '35er', '5': '46rt',
    '6': '57ty', '7': '68yu', '8': '79ui', '9': '80io', '0': '9-op',
    '`': '1', '-': '0=p', '=': '-[',
    '[': ']p', ']': '[\\', '\\': ']',
    ';': "l'", "'": ';',
    ',': 'm.', '.': ',/', '/': '.',
    ' ': 'zxcvbnm',  # Spacebar
    # Including shift-accessible characters
    '!': '@1', '@': '#!2', '#': '$@3', '$': '%#4', '%': '^$5',
    '^': '&%6', '&': '*^7', '*': '(&8', '(': ')*9', ')': '_)0',
    '_': '+_-', '+': '=',
    '{': '}[', '}': '{]|', '|': '}\\',
    ':': '"', '"': ':',
    '<': '>,', '>': '<?', '?': '>/'
}

In [6]:
internet_slang = [
    'lol', 'rofl', 'idk', 'tbh', 'imo', 'fyi', 'brb', 'afk', 'tl;dr',
    'wtf', 'omg', 'smh', 'yolo', 'fomo', 'irl', 'tfw', 'ftw', 'imho',
    'iirc', 'asap', 'thx', 'hmu', 'xoxo', 'fwiw', 'ftfy', 'ama', 'eli5'
]

In [7]:
stop_words_v1 = set([
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an', 'and', 'any', 'are', 'aren\'t', 'as', 'at',
    'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by',
    'can', 'can\'t', 'cannot', 'could', 'couldn\'t',
    'did', 'didn\'t', 'do', 'does', 'doesn\'t', 'doing', 'don\'t', 'down', 'during',
    'each',
    'few', 'for', 'from', 'further',
    'had', 'hadn\'t', 'has', 'hasn\'t', 'have', 'haven\'t', 'having', 'he', 'he\'d', 'he\'ll', 'he\'s', 'her', 'here', 'here\'s', 'hers', 'herself', 'him', 'himself', 'his', 'how', 'how\'s',
    'i', 'i\'d', 'i\'ll', 'i\'m', 'i\'ve', 'if', 'in', 'into', 'is', 'isn\'t', 'it', 'it\'s', 'its', 'itself',
    'let\'s',
    'me', 'more', 'most', 'mustn\'t', 'my', 'myself',
    'no', 'nor', 'not',
    'of', 'off', 'on', 'once', 'only', 'or', 'other', 'ought', 'our', 'ours', 'ourselves', 'out', 'over', 'own',
    'same', 'shan\'t', 'she', 'she\'d', 'she\'ll', 'she\'s', 'should', 'shouldn\'t', 'so', 'some', 'such',
    'than', 'that', 'that\'s', 'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there', 'there\'s', 'these', 'they', 'they\'d', 'they\'ll', 'they\'re', 'they\'ve', 'this', 'those', 'through', 'to', 'too',
    'under', 'until', 'up',
    'very',
    'was', 'wasn\'t', 'we', 'we\'d', 'we\'ll', 'we\'re', 'we\'ve', 'were', 'weren\'t', 'what', 'what\'s', 'when', 'when\'s', 'where', 'where\'s', 'which', 'while', 'who', 'who\'s', 'whom', 'why', 'why\'s', 'with', 'won\'t', 'would', 'wouldn\'t',
    'you', 'you\'d', 'you\'ll', 'you\'re', 'you\'ve', 'your', 'yours', 'yourself', 'yourselves',
    # Additional common words often considered as stop words
    'able', 'across', 'almost', 'always', 'among', 'another', 'anybody', 'anyone', 'anything', 'anywhere', 'around',
    'became', 'become', 'becomes', 'becoming', 'behind', 'beside', 'besides', 'beyond',
    'cannot', 'certain', 'certainly', 'come', 'comes', 'contain', 'containing', 'contains',
    'done', 'due', 'during',
    'either', 'else', 'elsewhere', 'enough', 'especially', 'etc', 'even', 'ever', 'every', 'everybody', 'everyone', 'everything', 'everywhere', 'except',
    'first', 'followed', 'following', 'follows', 'front', 'full', 'further',
    'gave', 'get', 'gets', 'getting', 'give', 'given', 'gives', 'giving', 'go', 'goes', 'going', 'gone', 'got', 'gotten',
    'happen', 'happens', 'hardly', 'has', 'have', 'having', 'hence', 'here', 'hereafter', 'hereby', 'herein', 'hereupon', 'however',
    'immediate', 'immediately', 'important', 'indeed', 'instead', 'into', 'inward',
    'just',
    'keep', 'keeps', 'kept', 'know', 'known', 'knows',
    'last', 'lately', 'later', 'latter', 'latterly', 'least', 'less', 'lest', 'like', 'liked', 'likely', 'little', 'look', 'looking', 'looks',
    'made', 'mainly', 'make', 'makes', 'many', 'may', 'maybe', 'mean', 'meanwhile', 'might', 'more', 'moreover', 'most', 'mostly', 'much', 'must',
    'name', 'namely', 'near', 'nearly', 'necessary', 'need', 'needs', 'neither', 'never', 'nevertheless', 'new', 'next', 'nine', 'nobody', 'none', 'noone', 'normally', 'nothing', 'now', 'nowhere',
    'obviously', 'often', 'okay', 'once', 'one', 'ones', 'onto', 'other', 'others', 'otherwise', 'ought',
    'perhaps', 'possible', 'probably',
    'quite',
    'rather', 'really', 'right',
    'said', 'saw', 'say', 'saying', 'says', 'second', 'secondly', 'see', 'seeing', 'seem', 'seemed', 'seeming', 'seems', 'seen', 'self', 'selves', 'sensible', 'sent', 'serious', 'seriously', 'seven', 'several', 'shall', 'since', 'six', 'somehow', 'someone', 'something', 'sometime', 'sometimes', 'somewhat', 'somewhere', 'soon', 'sorry', 'specified', 'specify', 'specifying', 'still', 'sub', 'sure',
    'take', 'taken', 'tell', 'tends', 'th', 'thank', 'thanks', 'thanx', 'that', 'thats', 'the', 'their', 'theirs', 'them', 'themselves', 'then', 'thence', 'there', 'thereafter', 'thereby', 'therefore', 'therein', 'theres', 'thereupon', 'these', 'they', 'think', 'third', 'this', 'thorough', 'thoroughly', 'those', 'though', 'three', 'through', 'throughout', 'thru', 'thus', 'together', 'too', 'took', 'toward', 'towards', 'tried', 'tries', 'truly', 'try', 'trying', 'twice', 'two',
    'un', 'under', 'unfortunately', 'unless', 'unlikely', 'until', 'unto', 'upon', 'use', 'used', 'useful', 'uses', 'using', 'usually',
    'value', 'various', 'very', 'via',
    'viz', 'vs',
    'want', 'wants', 'way', 'well', 'went', 'whatever', 'whence', 'whenever', 'whereafter', 'whereas', 'whereby', 'wherein', 'whereupon', 'wherever', 'whether', 'whither', 'whoever', 'whole', 'whose', 'why', 'will', 'willing', 'wish', 'within', 'without', 'wonder', 'would',
    'yes', 'yet', 'you', 'your', 'yours', 'yourself', 'yourselves'
])

stop_words_v2 = set([
    # Common phrases and expressions
    'in order to', 'as well as', 'as long as', 'in addition to', 'with regard to',
    'in terms of', 'on the other hand', 'in fact', 'as a result', 'for example',
    'in general', 'in particular', 'such as', 'rather than', 'as far as',
    'in case', 'in spite of', 'with respect to', 'due to the fact that', 'for the purpose of',
    
    # Internet and technology related
    'http', 'https', 'www', 'com', 'org', 'net', 'edu', 'gov',
    'email', 'website', 'webpage', 'online', 'offline', 'internet', 'web',
    'download', 'upload', 'login', 'logout', 'username', 'password',
    
    # Miscellaneous
    'etc', 'i.e', 'e.g', 'vs', 'versus', 'viz', 'namely', 'item', 'things',
    'stuff', 'aspect', 'feature', 'characteristic', 'quality', 'quantity',
    'approximately', 'roughly', 'about', 'around', 'circa',
    'actually', 'literally', 'basically', 'essentially', 'virtually',
    'presumably', 'supposedly', 'allegedly', 'apparently', 'seemingly'
])

stop_words = stop_words_v1.union(stop_words_v2)

In [15]:
import random
import string
import nltk
from nltk.corpus import wordnet
from nltk.corpus import words as nltk_words
from nltk import FreqDist
from difflib import SequenceMatcher
from googletrans import Translator
from transformers import RobertaTokenizer, RobertaForMaskedLM
import torch

# Set the custom download directory
nltk_data_dir = '/nfs/students/daro/data/nltk_data'

# Add the custom directory to NLTK's data path
nltk.data.path.append(nltk_data_dir)

# Download NLTK data if not already present
if not os.path.exists(os.path.join(nltk_data_dir, 'corpora', 'words')):
    nltk.download('all', download_dir=nltk_data_dir)

translator = Translator()

def load_file_to_dict(file_path):
    cmw_dict = {}
    with open(file_path, 'r') as file:
        for line in file:
            correct_word, incorrect_word = line.strip().split(':')
            cmw_dict[correct_word.strip()] = incorrect_word.strip()
    return cmw_dict

def random_phrase_translation(word_list, num_translations):
    """Translate random words or phrases to a random foreign language."""
    languages = ['es', 'fr', 'de', 'it', 'ru', 'zh-cn', 'ja']
    for _ in range(num_translations):
        if len(word_list) < 2:
            return word_list
        
        is_phrase = random.choice([True, False]) if len(word_list) > 2 else False
        
        try:
            if is_phrase:
                start_index = random.randint(0, len(word_list) - 2)
                phrase = ' '.join(word_list[start_index:start_index+2])
                lang = random.choice(languages)
                translated_phrase = translator.translate(phrase, dest=lang).text
                word_list[start_index:start_index+2] = translated_phrase.split()
            else:
                word_idx = random.randint(0, len(word_list) - 1)
                lang = random.choice(languages)
                translated_word = translator.translate(word_list[word_idx], dest=lang).text
                word_list[word_idx] = translated_word
        except Exception as e:
            print(f"Translation error: {e}")
    
    return word_list

def random_insertion(word, num_insertions):
    for _ in range(num_insertions):
        pos = random.randint(0, len(word))
        char_to_insert = random.choice(string.ascii_letters)
        word = word[:pos] + char_to_insert + word[pos:]
    return word

def random_deletion(word, num_deletions):
    for _ in range(num_deletions):
        if len(word) > 1:
            pos = random.randint(0, len(word) - 1)
            word = word[:pos] + word[pos+1:]
    return word

def random_replacement(word, num_replacements):
    for _ in range(num_replacements):
        if len(word) > 0:
            pos = random.randint(0, len(word) - 1)
            if word[pos].lower() in keyboard_adjacency:
                replacement_char = random.choice(keyboard_adjacency[word[pos].lower()])
                word = word[:pos] + replacement_char + word[pos+1:]
    return word

def random_repetition(word, num_repetitions):
    for _ in range(num_repetitions):
        pos = random.randint(0, len(word))
        word = word[:pos] + word[pos-1:pos] + word[pos:]
    return word

def random_swapping(word, num_swaps):
    for _ in range(num_swaps):
        if len(word) > 1:
            pos = random.randint(0, len(word) - 2)
            word = word[:pos] + word[pos+1] + word[pos] + word[pos+2:]
    return word

def apply_cmw(word_list, num_replacements, cmw_dict):
    misspellable_words = [word for word in word_list if word.lower() in cmw_dict]
    num_replacements = min(num_replacements, len(misspellable_words))
    for _ in range(num_replacements):
        if misspellable_words:
            word_to_misspell = random.choice(misspellable_words)
            index = word_list.index(word_to_misspell)
            word_list[index] = cmw_dict[word_to_misspell.lower()]
            misspellable_words.remove(word_to_misspell)
    return word_list

def random_letter_case(word, num_case_changes):
    for _ in range(num_case_changes):
        if len(word) > 0:
            pos = random.randint(0, len(word) - 1)
            word = word[:pos] + word[pos].swapcase() + word[pos+1:]
    return word

def synonym_replacement(word_list, num_replacements):
    def get_homophones(word):
        homophones = []
        for synset in wordnet.synsets(word):
            for lemma in synset.lemmas():
                if lemma.name() != word and lemma.name().lower() not in homophones:
                    homophones.append(lemma.name().lower())
        return homophones

    # Create a frequency distribution of words in the NLTK words corpus
    word_freq = FreqDist(word.lower() for word in nltk_words.words())

    def get_valid_homophone(word, homophones):
        sorted_homophones = sorted(homophones, key=lambda x: word_freq[x], reverse=True)
        for homophone in sorted_homophones:
            if not is_similar(word, homophone):
                print(f"Replaced '{word}' with '{homophone}'")
                return homophone
            else:
                print(f"Skipped '{word}' and '{homophone}'")
        return None

    def is_similar(word1, word2):
        # Check if the words differ by only one character
        if word1.lower() in word2.lower() or word2.lower() in word1.lower():
            return True
        if abs(len(word1) - len(word2)) <= 1:
            matcher = SequenceMatcher(None, word1.lower(), word2.lower())
            return matcher.ratio() > 0.8  # Adjust this threshold as needed
        return False

    replacements_made = 0
    replaced_indices = set()

    for i, word in enumerate(word_list):
        if replacements_made >= num_replacements:
            break
        if i in replaced_indices:
            continue

        homophones = get_homophones(word)
        if homophones:
            valid_homophone = get_valid_homophone(word, homophones)
            if valid_homophone:
                word_list[i] = valid_homophone
                replaced_indices.add(i)
                replacements_made += 1

    return word_list

def context_aware_insertion(word_list, num_insertions):
    # Load pre-trained RoBERTa model and tokenizer
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    model = RobertaForMaskedLM.from_pretrained("roberta-base")
    model.eval()

    # Convert word list to sentence
    sentence = " ".join(word_list)

    for _ in range(num_insertions):
        # Choose a random position to insert a word
        insert_position = random.randint(0, len(word_list))

        # Create a masked sentence for prediction
        masked_sentence = word_list[:insert_position] + [tokenizer.mask_token] + word_list[insert_position:]
        masked_sentence = " ".join(masked_sentence)

        # Tokenize and get model predictions
        inputs = tokenizer(masked_sentence, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**inputs)

        # Get the predicted token
        mask_token_index = torch.where(inputs["input_ids"][0] == tokenizer.mask_token_id)[0]
        predicted_token_id = outputs.logits[0, mask_token_index].argmax(axis=-1)
        predicted_token = tokenizer.decode(predicted_token_id)

        # Insert the predicted word if it's not empty
        if predicted_token.strip() != "":
            word_list.insert(insert_position, predicted_token.strip())

    return word_list

def add_noise_characters(word, num_noises):
    for _ in range(num_noises):
        pos = random.randint(0, len(word))
        noise_char = random.choice(string.punctuation + string.digits)
        word = word[:pos] + noise_char + word[pos:]
    return word

def add_taxonomy(query, taxonomy, num_confusions):
    for _ in range(num_confusions):
        if taxonomy:
            random_taxonomy = random.choice(taxonomy)
            query = f"{random_taxonomy}. {query}"
    return query

def repeat_key_words(word_list, num_repetitions):
    for _ in range(num_repetitions):
        if word_list:
            word_idx = random.randint(0, len(word_list) - 1)
            word_list.insert(word_idx, word_list[word_idx])
    return word_list

def random_char_substitution(word, num_substitutions):
    for _ in range(num_substitutions):
        if len(word) > 0:
            pos = random.randint(0, len(word) - 1)
            if word[pos].upper() in char_map:
                word = word[:pos] + char_map[word[pos].upper()] + word[pos+1:]
    return word

def internet_slang_insertion(word_list, num_insertions):
    for _ in range(num_insertions):
        pos = random.randint(0, len(word_list))
        word_list.insert(pos, random.choice(internet_slang))
    return word_list

def emoji_substitution(word_list, num_substitutions):
    for _ in range(num_substitutions):
        substitutable_words = [(i, word.lower()) for i, word in enumerate(word_list) if word.lower() in emoji_dict]
        if not substitutable_words:
            break
        index, word = random.choice(substitutable_words)
        word_list[index] = emoji_dict[word]
    return word_list

def remove_punctuation_capitalization(query):
    return ''.join(char.lower() for char in query if char.isalnum() or char.isspace())

def keyword_only_query(query):
    return ' '.join([word for word in query.split() if word.lower() not in stop_words])

def apply_typo_modifications(query, typo_dict, taxonomy=[]):
    cmw_file_path = 'data/cmw_v2.txt'
    cmw_dict = load_file_to_dict(cmw_file_path)
    words = query.split()
    
    for mod_type, num_modifications in typo_dict.items():
        if mod_type in ['remove_punctuation', 'keyword_only']:
            continue  # These are handled separately at the end
        
        if num_modifications > 0:
            if mod_type == 'CMW':
                words = apply_cmw(words, num_modifications, cmw_dict)
            elif mod_type == 'synonym':
                words = synonym_replacement(words, num_modifications)
            elif mod_type in ['noise', 'char_substitution', 'insertion', 'deletion', 'replacement', 'repetition', 'swapping', 'LCC']:
                indices = random.sample(range(len(words)), min(num_modifications, len(words)))
                for idx in indices:
                    if mod_type == 'noise':
                        words[idx] = add_noise_characters(words[idx], 1)
                    elif mod_type == 'char_substitution':
                        words[idx] = random_char_substitution(words[idx], 1)
                    elif mod_type == 'insertion':
                        words[idx] = random_insertion(words[idx], 1)
                    elif mod_type == 'deletion':
                        words[idx] = random_deletion(words[idx], 1)
                    elif mod_type == 'replacement':
                        words[idx] = random_replacement(words[idx], 1)
                    elif mod_type == 'repetition':
                        words[idx] = random_repetition(words[idx], 1)
                    elif mod_type == 'swapping':
                        words[idx] = random_swapping(words[idx], 1)
                    elif mod_type == 'LCC':
                        words[idx] = random_letter_case(words[idx], 1)
            elif mod_type == 'emoji':
                words = emoji_substitution(words, num_modifications)
            elif mod_type == 'internet_slang':
                words = internet_slang_insertion(words, num_modifications)
            elif mod_type == 'phrase_translation':
                words = random_phrase_translation(words, num_modifications)
            elif mod_type == 'confusion':
                query = add_taxonomy(query, taxonomy, num_modifications)
                words = query.split()  # Update words list after taxonomy modification
            elif mod_type == 'repeat':
                words = repeat_key_words(words, num_modifications)
    
    modified_query = ' '.join(words)
    
    if typo_dict.get('remove_punctuation', False):
        modified_query = remove_punctuation_capitalization(modified_query)
    if typo_dict.get('keyword_only', False):
        modified_query = keyword_only_query(modified_query)
    
    return modified_query

# List of 10 diverse and short questions
questions = [
    "What is the capital of France?",
    "Who wrote 'Romeo and Juliet'?",
    "What is the chemical symbol for gold?",
    "How many planets are in our solar system?",
    "What year did World War II end?",
    "What is the largest mammal on Earth?",
    "Who painted the Mona Lisa?",
    "What is the square root of 64?",
    "Which country is known as the Land of the Rising Sun?",
    "What is the main ingredient in guacamole?"
]

# Example taxonomies for each question
taxonomies = [
    ["Paris", "Lyon", "Marseille"],
    ["William Shakespeare", "Charles Dickens", "Jane Austen"],
    ["Au", "Ag", "Fe"],
    ["8", "9", "7"],
    ["1945", "1944", "1946"],
    ["Blue Whale", "African Elephant", "Giraffe"],
    ["Leonardo da Vinci", "Michelangelo", "Raphael"],
    ["8", "7", "9"],
    ["Japan", "China", "Korea"],
    ["Avocado", "Tomato", "Onion"]
]

typo_dict = {
    "insertion": 0,
    "deletion": 0,
    "replacement": 0,
    "repetition": 0,
    "swapping": 0,
    "CMW": 0,
    "LCC": 0,
    "synonym": 3,
    "noise": 0,
    "confusion": 0,
    "repeat": 0,
    "char_substitution": 0,
    "emoji": 0,
    "internet_slang": 0,
    "phrase_translation": 0,
    "remove_punctuation": False,  # to be randomly applied
    "keyword_only": False  # to be randomly applied
}

if __name__ == "__main__":
    for i, query in enumerate(questions):
        if i >= 100:
            break
        print(f"\nQuestion {i+1}:")
        print("Original query:")
        print(query)
        
        modified_query = apply_typo_modifications(query, typo_dict, taxonomies[i])
        
        print("Modified query:")
        print(modified_query)


Question 1:
Original query:
What is the capital of France?
Replaced 'is' with 'be'
Replaced 'capital' with 'great'
Modified query:
What be the great of France?

Question 2:
Original query:
Who wrote 'Romeo and Juliet'?
Skipped 'Who' and 'who'
Replaced 'Who' with 'world_health_organization'
Replaced 'wrote' with 'pen'
Modified query:
world_health_organization pen 'Romeo and Juliet'?

Question 3:
Original query:
What is the chemical symbol for gold?
Replaced 'is' with 'be'
Replaced 'chemical' with 'chemic'
Replaced 'symbol' with 'symbolization'
Modified query:
What be the chemic symbolization for gold?

Question 4:
Original query:
How many planets are in our solar system?
Skipped 'planets' and 'planet'
Replaced 'planets' with 'satellite'
Replaced 'are' with 'be'
Skipped 'in' and 'in'
Replaced 'in' with 'inch'
Modified query:
How many satellite be inch our solar system?

Question 5:
Original query:
What year did World War II end?
Replaced 'year' with 'twelvemonth'
Replaced 'did' with 'ma

### Test modifications

In [25]:
import random

def test_typo_modifications():
    # Example query
    query = "What is the capital of France?"
    # Taxonomy for confusion modification
    taxonomy = ["Paris", "London", "Berlin", "Rome", "Madrid"]
    # List of all implemented modification types
    modification_types = [
        "insertion", "deletion", "replacement", "repetition", "swapping",
        "CMW", "LCC", "synonym", "noise", "confusion", "repeat",
        "char_substitution", "emoji", "internet_slang", "phrase_translation",
        "remove_punctuation", "keyword_only"
    ]

    print("Typo Modification Tester")
    print("========================")
    print(f"Original query: {query}\n")

    # Apply each modification type individually
    for mod_type in modification_types:
        print(f"Applying {mod_type}:")
        for intensity in range(1, 6):  # Test intensities 1 to 5
            typo_dict = {mod: 0 for mod in modification_types if mod not in ["remove_punctuation", "keyword_only"]}
            if mod_type in ["remove_punctuation", "keyword_only"]:
                typo_dict[mod_type] = True if intensity == 1 else False
            else:
                typo_dict[mod_type] = intensity
            modified_query = apply_typo_modifications(query, typo_dict, taxonomy)
            print(f" Intensity {intensity}: {modified_query}")
        print()

    # Apply all modifications together
    print("Applying all modifications:")
    all_mods_dict = {mod: 2 for mod in modification_types if mod not in ["remove_punctuation", "keyword_only"]}
    all_mods_dict.update({"remove_punctuation": True, "keyword_only": True})
    for _ in range(5):  # Generate 5 examples with all modifications
        modified_query = apply_typo_modifications(query, all_mods_dict, taxonomy)
        print(f" {modified_query}")

if __name__ == "__main__":
    test_typo_modifications()

Typo Modification Tester
Original query: What is the capital of France?

Applying insertion:
 Intensity 1: bWhat is the capital of France?
 Intensity 2: Whajt is the capital ofl France?
 Intensity 3: What is tdhe capGital oSf France?
 Intensity 4: Whact isz Mthe capital of OFrance?
 Intensity 5: WhMat is tXhe cappital Hof FrancQe?

Applying deletion:
 Intensity 1: What i the capital of France?
 Intensity 2: What is the capita o France?
 Intensity 3: What is he capitl of Fance?
 Intensity 4: What is te apital o France
 Intensity 5: What i th apital o Frane?

Applying replacement:
 Intensity 1: What is ths capital of France?
 Intensity 2: What us the capigal of France?
 Intensity 3: 2hat is 5he fapital of France?
 Intensity 4: Wkat ls the cspital of France/
 Intensity 5: Whst 8s tke vapital oc France?

Applying repetition:
 Intensity 1: What is the capital of FFrance?
 Intensity 2: What is the capital of Francee?
 Intensity 3: What is thhe capitaal of Fraance?
 Intensity 4: What is thee 

In [ ]:
modifications_to_exclude = [
  "language_switch"
]

modifications_needing_improvement = [
  "remove_punctuation",
  "internet_slang",
  "keyword_only",
  "autocorrect",
  "tts_mishearing",
  "emoji",
]

## 4. Semantic Similarity

In [ ]:
from gensim.models import KeyedVectors
from nlpia.data.loaders import get_data, BIGDATA_PATH

wordvector_path = os.path.join(BIGDATA_PATH, 'GoogleNews-vectors-negative300.bin.gz')

# Load a pre-trained word2vec model (this is just an example, the path should be to your downloaded model)
word_vectors = KeyedVectors.load_word2vec_format(wordvector_path, binary=True)

# Function to find similar words
def find_similar_words(word, topn=10):
    try:
        similar_words = word_vectors.most_similar(positive=[word], topn=topn)
        return [word for word, similarity in similar_words]
    except KeyError:
        return []

# Example usage
related_words = find_similar_words('philosopher')
print(related_words)

ImportError: cannot import name 'Mapping' from 'collections' (/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/collections/__init__.py)

In [ ]:
from nltk.corpus import wordnet as wn
import nltk

# Download WordNet data
nltk.download('wordnet')

def find_related_nouns(word):
    related_nouns = set()
    for synset in wn.synsets(word, pos=wn.NOUN):
        # Traverse through the hyponyms (subordinate concepts) and hypernyms (superordinate concepts)
        for lemma in synset.lemmas():
            related_nouns.add(lemma.name())
        for hypernym in synset.hypernyms():
            for lemma in hypernym.lemmas():
                related_nouns.add(lemma.name())
    return related_nouns

# Example usage
related_words = find_related_nouns('philosopher')
print(related_words)

[nltk_data] Downloading package wordnet to
[nltk_data]     /nfs/homedirs/daro/nltk_data...


{'student', 'scholar', 'scholarly_person', 'soul', 'individual', 'bookman', 'philosopher', 'person', 'somebody', 'mortal', 'someone'}


In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load a pre-trained sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Function to find semantically similar words
def find_similar_phrases(word, context_word):
    # Encode word and context into embeddings
    word_embedding = model.encode(word)
    context_embedding = model.encode(context_word)

    # Compute cosine similarity
    similarity_score = util.pytorch_cos_sim(word_embedding, context_embedding).item()
    
    return similarity_score

# Example usage
similarity = find_similar_phrases('philosopher', 'philosophy')
print(similarity)

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


0.8044580817222595


## 5. Answer Correctness Handling

In [ ]:
from fuzzywuzzy import fuzz

# Example 1: Misspelling
str1 = "philosophy"
str2 = "filosophy"
ratio = fuzz.ratio(str1, str2)
print(f"Similarity between '{str1}' and '{str2}': {ratio}%")

# Example 2: Word Order Change
str3 = "John Smith from New York"
str4 = "Smith, John - New York"
ratio = fuzz.token_sort_ratio(str3, str4)
print(f"Similarity between '{str3}' and '{str4}': {ratio}%")

# Bonus: Partial String Matching
str5 = "The quick brown fox jumps over the lazy dog"
str6 = "brown fox"
ratio = fuzz.partial_ratio(str5, str6)
print(f"Partial match of '{str6}' in '{str5}': {ratio}%")

Similarity between 'philosophy' and 'filosophy': 84%
Similarity between 'John Smith from New York' and 'Smith, John - New York': 88%
Partial match of 'brown fox' in 'The quick brown fox jumps over the lazy dog': 100%


## 6. Generate PDF

In [4]:
import random
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_JUSTIFY

def summarize_modifications():
    modifications = [
        {
            "name": "Random Insertion",
            "description": "Randomly inserts characters into a word.",
            "implementation": "Selects a random position in the word and inserts a random letter.",
            "example": ("Where is the capital of Germany?", "Where is thea capital of Germany?")
        },
        {
            "name": "Random Deletion",
            "description": "Randomly deletes characters from a word.",
            "implementation": "Selects a random position in the word and removes the character at that position.",
            "example": ("Where is the capital of Germany?", "Where is th capital of Germany?")
        },
        {
            "name": "Random Replacement",
            "description": "Randomly replaces characters in a word with adjacent keys on the keyboard.",
            "implementation": "Selects a random character in the word and replaces it with a random adjacent key on the keyboard.",
            "example": ("Where is the capital of Germany?", "Where is the capitsl of Germany?")
        },
        {
            "name": "Random Repetition",
            "description": "Randomly repeats characters in a word.",
            "implementation": "Selects a random position in the word and duplicates the character at that position.",
            "example": ("Where is the capital of Germany?", "Where is the capittal of Germany?")
        },
        {
            "name": "Random Swapping",
            "description": "Randomly swaps adjacent characters in a word.",
            "implementation": "Selects a random position in the word and swaps the character with its adjacent character.",
            "example": ("Where is the capital of Germany?", "Where is hte capital of Germany?")
        },
        {
            "name": "Common Misspelling Words (CMW)",
            "description": "Replaces words with their common misspelled variants.",
            "implementation": "Checks if a word is in a predefined dictionary of common misspellings and replaces it if found.",
            "example": ("Where is the capital of Germany?", "Where is the capitol of Germany?")
        },
        {
            "name": "Random Letter Case Change",
            "description": "Randomly changes the case of letters in a word.",
            "implementation": "Selects a random letter in the word and changes its case (upper to lower or vice versa).",
            "example": ("Where is the capital of Germany?", "Where is the caPital of Germany?")
        },
        {
            "name": "Synonym Replacement",
            "description": "Replaces words with their synonyms.",
            "implementation": "Uses NLTK's WordNet to find synonyms for a randomly selected word and replaces it.",
            "example": ("Where is the capital of Germany?", "Where is the metropolis of Germany?")
        },
        {
            "name": "Add Noise Characters",
            "description": "Adds random noise characters into words.",
            "implementation": "Inserts random punctuation or digit characters into a randomly selected word.",
            "example": ("Where is the capital of Germany?", "Where is the cap1ital of Germany?")
        },
        {
            "name": "Add Taxonomy",
            "description": "Adds a confusing taxonomy choice at the start of the query.",
            "implementation": "Prepends a random item from a predefined taxonomy list to the query.",
            "example": ("Where is the capital of Germany?", "Munich. Where is the capital of Germany?")
        },
        {
            "name": "Repeat Key Words",
            "description": "Repeats key words in the questions.",
            "implementation": "Selects a random word in the query and duplicates it at a random position.",
            "example": ("Where is the capital of Germany?", "Where is the capital capital of Germany?")
        },
        {
            "name": "Language Switching",
            "description": "Inserts words from a foreign language.",
            "implementation": "Inserts a random foreign word from a predefined list into the query.",
            "example": ("Where is the capital of Germany?", "Where is the hola capital of Germany?")
        },
        {
            "name": "Random Character Substitution",
            "description": "Substitutes characters with visually similar numbers or symbols.",
            "implementation": "Replaces certain letters with visually similar numbers or symbols (e.g., 'O' with '0').",
            "example": ("Where is the capital of Germany?", "Where is the capit@l of Germany?")
        },
        {
            "name": "Random Foreign Word Insertion",
            "description": "Inserts random foreign words into the word list.",
            "implementation": "Inserts a random foreign word from a predefined list into the query.",
            "example": ("Where is the capital of Germany?", "Where is the bonjour capital of Germany?")
        },
        {
            "name": "Random Phrase Translation",
            "description": "Translates random words or phrases to a random foreign language and back.",
            "implementation": "Uses a translation API to translate a random word to a foreign language and back.",
            "example": ("Where is the capital of Germany?", "Where is the Hauptstadt of Germany?")
        },
        {
            "name": "Language-based Structural Transformation",
            "description": "Transforms sentence structure to reflect foreign language syntax.",
            "implementation": "Rearranges the word order based on the syntax of a randomly chosen foreign language.",
            "example": ("Where is the capital of Germany?", "Where Germany of the capital is?")
        },
        {
            "name": "Emoji Substitution",
            "description": "Replace words with related emojis or insert emojis into the query.",
            "implementation": "1. Create a dictionary mapping words to related emojis.\n2. Randomly select words in the query.\n3. Replace selected words with their emoji equivalents or insert emojis after them.",
            "example": ("What is the weather like today?", "What is the ☀️ like today? 🌤️")
        },
        {
            "name": "Text-to-Speech Mishearing Simulation",
            "description": "Modify words to simulate common speech recognition errors.",
            "implementation": "1. Create a list of common homophones and near-homophones.\n2. Scan the query for words that have homophones.\n3. Randomly replace words with their homophones.",
            "example": ("How to pair my Bluetooth device?", "How to pear my Bluetooth device?")
        },
        {
            "name": "Autocorrect Gone Wrong",
            "description": "Simulate autocorrect mistakes by replacing words with similarly spelled but incorrect words.",
            "implementation": "1. Create a dictionary of common autocorrect mistakes.\n2. Scan the query for words that match keys in the dictionary.\n3. Replace matching words with their incorrect 'autocorrected' versions.",
            "example": ("How to make duck confit?", "How to make duck confident?")
        },
        {
            "name": "Internet Slang Insertion",
            "description": "Insert or replace words with common internet slang and abbreviations.",
            "implementation": "1. Create a dictionary of internet slang terms and their meanings.\n2. Randomly select words in the query.\n3. Replace selected words with their slang equivalents or insert slang terms.",
            "example": ("What are the best movies to watch?", "What are the best movies to watch? IMHO TBH")
        },
        {
            "name": "Punctuation and Capitalization Removal",
            "description": "Remove punctuation and capitalization to mimic quick, informal typing.",
            "implementation": "1. Remove all punctuation marks from the query.\n2. Convert the entire query to lowercase.",
            "example": ("What's the capital of France? Is it Paris?", "whats the capital of france is it paris")
        },
        {
            "name": "Keyword-Only Query",
            "description": "Reduce the query to essential keywords, simulating users who type minimal search terms.",
            "implementation": "1. Identify stop words (common words like 'the', 'is', 'are').\n2. Remove stop words and retain only key terms.",
            "example": ("What are the symptoms of the common cold?", "symptoms common cold")
        },
        {
            "name": "Symbolic Substitution",
            "description": "Replace words or parts of words with symbols or emojis that visually resemble them.",
            "implementation": "1. Create a dictionary mapping letters or words to visually similar symbols or emojis.\n2. Scan the query for matches in the dictionary.\n3. Replace matches with their symbolic or emoji counterparts.",
            "example": ("How to solve for x in algebra?", "H♡w t♡ s♡lve f♡r ✖ in ∀lgebra? 🧮")
        }
    ]
    return modifications

def create_pdf(filename):
    doc = SimpleDocTemplate(filename, pagesize=letter,
                            rightMargin=72, leftMargin=72,
                            topMargin=72, bottomMargin=18)
    story = []
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name='Justify', alignment=TA_JUSTIFY))

    title = Paragraph("Typo and Query Modification Strategies", styles['Title'])
    story.append(title)
    story.append(Spacer(1, 12))

    for mod in summarize_modifications():
        name = Paragraph(f"<b>{mod['name']}</b>", styles['Heading2'])
        story.append(name)
        story.append(Spacer(1, 6))

        description = Paragraph(f"<b>Description:</b> {mod['description']}", styles['Normal'])
        story.append(description)
        story.append(Spacer(1, 6))

        implementation = Paragraph(f"<b>Implementation:</b> {mod['implementation']}", styles['Normal'])
        story.append(implementation)
        story.append(Spacer(1, 6))

        example = Paragraph(f"<b>Example:</b>", styles['Normal'])
        story.append(example)
        story.append(Spacer(1, 6))

        original = Paragraph(f"Original: {mod['example'][0]}", styles['Normal'])
        story.append(original)
        story.append(Spacer(1, 6))

        modified = Paragraph(f"Modified: {mod['example'][1]}", styles['Normal'])
        story.append(modified)
        story.append(Spacer(1, 12))

    doc.build(story)

if __name__ == "__main__":
    create_pdf("results/reliability_eval/typo_modification_summary.pdf")
    print("PDF report generated: typo_modification_summary.pdf")

PDF report generated: typo_modification_summary.pdf
